In [ ]:
import re
import pandas as pd

# 1. Leer el archivo original de 2021-2020 como texto plano
with open('llegadas_2021_2020_clean.csv', 'r', encoding='utf-8-sig', errors='ignore') as f:
    lineas = f.readlines()

lineas_datos = [l.strip() for l in lineas if l.strip() != ""]
if "RO" in lineas_datos[0]:
    lineas_datos = lineas_datos[1:]

# 2. ORDEN INVERTIDO DE LAS COLUMNAS (De más nuevo a más viejo: Dic 2021 -> Ene 2020)
meses_columnas = [
    "2021-12", "2021-11", "2021-10", "2021-09", "2021-08", "2021-07",
    "2021-06", "2021-05", "2021-04", "2021-03", "2021-02", "2021-01",
    "2020-12", "2020-11", "2020-10", "2020-09", "2020-08", "2020-07",
    "2020-06", "2020-05", "2020-04", "2020-03", "2020-02", "2020-01"
]

matriz_paises = {}

# 3. Extraer la serie temporal de cada país
for linea in lineas_datos[1:]:
    if "Pasajeros Totales" in linea:
        continue

    paises_encontrados = re.findall(r'[A-ZÁÉÍÓÚÑüÜ]{3,}(?:\s+[A-ZÁÉÍÓÚÑüÜ]+)*', linea)
    if not paises_encontrados:
        continue
    pais = paises_encontrados[0].strip()

    numeros = re.findall(r'"([\d\.]+)"', linea)
    if not numeros:
        numeros = re.findall(r'\b\d+(?:\.\d+)*\b', linea)

    valores_meses = []
    for num in numeros:
        num_limpio = num.replace('.', '')
        if num_limpio.isdigit():
            valores_meses.append(int(num_limpio))

    matriz_paises[pais] = valores_meses

# 4. Construir la estructura temporal mes a mes
paises_top = {'ESPAÑA', 'SPAIN', 'FRANCIA', 'FRANCE', 'ITALIA', 'ITALY',
              'REINO UNIDO', 'UNITED KINGDOM', 'UK', 'PAISES BAJOS', 'HOLANDA',
              'NETHERLANDS', 'ALEMANIA', 'GERMANY'}

datos_historicos_finales = []

for idx_mes, mes_nombre in enumerate(meses_columnas):

    def extraer_valor_mes(nombre_pais):
        for k, lista_valores in matriz_paises.items():
            if nombre_pais in k and idx_mes < len(lista_valores):
                return lista_valores[idx_mes]
        return 0

    nac = extraer_valor_mes('ESPAÑA') or extraer_valor_mes('SPAIN')
    fr = extraer_valor_mes('FRANCIA') or extraer_valor_mes('FRANCE')
    it = extraer_valor_mes('ITALIA') or extraer_valor_mes('ITALY')
    uk = extraer_valor_mes('REINO UNIDO') or extraer_valor_mes('UNITED KINGDOM') or extraer_valor_mes('UK')
    nl = extraer_valor_mes('PAISES BAJOS') or extraer_valor_mes('HOLANDA') or extraer_valor_mes('NETHERLANDS')
    de = extraer_valor_mes('ALEMANIA') or extraer_valor_mes('GERMANY')

    resto_del_mundo = 0
    for k, lista_valores in matriz_paises.items():
        if not any(p in k for p in paises_top) and idx_mes < len(lista_valores):
            resto_del_mundo += lista_valores[idx_mes]

    internacionales = 0
    for k, lista_valores in matriz_paises.items():
        if 'ESPAÑA' not in k and 'SPAIN' not in k and idx_mes < len(lista_valores):
            internacionales += lista_valores[idx_mes]

    total_aeropuerto = nac + internacionales

    datos_historicos_finales.append({
        'Mes': mes_nombre,
        'Volumen total llegadas aeropuerto': total_aeropuerto,
        'Volumen llegadas Nacionales': nac,
        'Volumen llegadas Internacionales': internacionales,
        'Volumen llegadas Francia': fr,
        'Volumen llegadas Italia': it,
        'Volumen llegadas Reino Unido': uk,
        'Volumen llegadas Países Bajos': nl,
        'Volumen llegadas Alemania': de,
        'Volumen llegadas Resto del Mundo': resto_del_mundo
    })

# 5. Convertir a DataFrame, ordenar cronológicamente (Ene 2020 -> Dic 2021) y guardar
df_temporal = pd.DataFrame(datos_historicos_finales)
df_temporal = df_temporal.sort_values(by='Mes').reset_index(drop=True)

# Guardar con el nombre correspondiente
df_temporal.to_csv('2021-2020.csv', index=False, encoding='utf-8-sig')


¡Listo! Tu archivo procesado ha sido guardado como: '2021-2020.csv'


,Mes,Volumen total llegadas aeropuerto,Volumen llegadas Nacionales,Volumen llegadas Internacionales,Volumen llegadas Francia,Volumen llegadas Italia,Volumen llegadas Reino Unido,Volumen llegadas Países Bajos,Volumen llegadas Alemania,Volumen llegadas Resto del Mundo
0,2020-01,263871,132640,131231,26790,30512,23053,7226,8447,35203
1,2020-02,286000,138048,147952,33156,27694,29377,9642,10152,37931
2,2020-03,122294,62165,60129,14412,4374,15213,5564,4822,15744
3,2020-04,1018,1003,15,0,0,6,2,0,7
4,2020-05,2466,2456,10,2,0,8,0,0,0
5,2020-06,12467,10534,1933,439,3,460,626,0,405
6,2020-07,96745,58035,38710,9715,5923,6919,3831,3432,8890
7,2020-08,124209,84660,39549,12087,6730,4020,2308,3332,11072
8,2020-09,88889,67189,21700,8184,2020,2879,1039,2221,5357
9,2020-10,79361,58396,20965,11350,990,2407,1181,1336,3701
